In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2023_Pusa_Delhi_DPCC_2023.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,313.0,198.0,193.0,96.0,160.0,120.0,51.0,108.0,169.0,177.0,365.0,362.0
1,2,357.0,237.0,233.0,115.0,70.0,215.0,53.0,119.0,170.0,160.0,358.0,361.0
2,3,388.0,238.0,196.0,148.0,111.0,192.0,NaN,108.0,166.0,165.0,477.0,314.0
3,4,368.0,250.0,159.0,107.0,86.0,192.0,141.0,122.0,151.0,215.0,386.0,320.0
4,5,361.0,279.0,173.0,189.0,193.0,199.0,83.0,90.0,122.0,202.0,446.0,302.0
5,6,413.0,157.0,156.0,177.0,220.0,141.0,62.0,133.0,101.0,203.0,414.0,299.0
6,7,NaN,257.0,158.0,179.0,172.0,267.0,51.0,124.0,98.0,172.0,382.0,295.0
7,8,416.0,164.0,191.0,217.0,120.0,202.0,38.0,139.0,79.0,153.0,428.0,317.0
8,9,446.0,235.0,116.0,206.0,215.0,186.0,NaN,143.0,35.0,159.0,435.0,317.0
9,10,429.0,146.0,177.0,214.0,246.0,168.0,NaN,153.0,28.0,NaN,264.0,323.0


In [4]:
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        40 non-null     object 
 1   January    36 non-null     float64
 2   February   32 non-null     float64
 3   March      35 non-null     float64
 4   April      34 non-null     float64
 5   May        36 non-null     float64
 6   June       34 non-null     float64
 7   July       22 non-null     float64
 8   August     34 non-null     float64
 9   September  34 non-null     float64
 10  October    35 non-null     float64
 11  November   34 non-null     float64
 12  December   34 non-null     float64
dtypes: float64(12), object(1)
memory usage: 4.3+ KB


In [5]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [6]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [7]:
# Define a function for outlier handling
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            # Replace outliers with mean
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [8]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready.head()

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,313.0,198.0,193.0,96.0,160.0,120.0,51.000000,108.0,169.0,177.0,365.0,362.0
1,2,357.0,237.0,233.0,115.0,70.0,215.0,53.000000,119.0,170.0,160.0,358.0,361.0
2,3,388.0,238.0,196.0,148.0,111.0,192.0,60.181818,108.0,166.0,165.0,477.0,314.0
3,4,368.0,250.0,159.0,107.0,86.0,192.0,60.181818,122.0,151.0,215.0,386.0,320.0
4,5,361.0,279.0,173.0,189.0,193.0,199.0,60.181818,90.0,122.0,202.0,446.0,302.0
